In [1]:
import pandas as pd
import numpy as np

In [2]:
participants = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106],
    "age": [24, 31, 27, 19, 42, 35],
    "group": ["A", "B", "A", "B", "A", "B"]
})

In [3]:
scores = pd.DataFrame({
    "id": [101, 102, 103, 105, 106, 107],
    "score": [81, 94, 88, 91, 85, 79]
})

In [5]:
participants


,id,age,group
0,101,24,A
1,102,31,B
2,103,27,A
3,104,19,B
4,105,42,A
5,106,35,B


In [6]:
scores

,id,score
0,101,81
1,102,94
2,103,88
3,105,91
4,106,85
5,107,79


1. 101,102,103,105,106 appear in both
2. 104 only appears in participants
3. 107 only appears in scores
4. An inner join will contain 101,102,103,105, and 106

In [7]:
inner = participants.merge(
    scores,
    on="id",
    how="inner"
)

In [9]:
inner

,id,age,group,score
0,101,24,A,81
1,102,31,B,94
2,103,27,A,88
3,105,42,A,91
4,106,35,B,85


In [10]:
inner.shape

(5, 4)

1. 104 and 107 were removed.
2. This is because an INNER join joins everything that is in both dataframes.
3. Yes it did.

In [11]:
left = participants.merge(
    scores,
    on="id",
    how="left"
)

In [12]:
left

,id,age,group,score
0,101,24,A,81.0
1,102,31,B,94.0
2,103,27,A,88.0
3,104,19,B,NaN
4,105,42,A,91.0
5,106,35,B,85.0


1. The left table (participants) defines the population. 
2. Because ID 104 does not have a score reported in the scores dataframe, the score will be missing. 
3. 104 is not referenced in the scores dataframe. 

In [13]:
outer = participants.merge(
    scores,
    on="id",
    how="outer"
)

In [14]:
outer

,id,age,group,score
0,101,24.0,A,81.0
1,102,31.0,B,94.0
2,103,27.0,A,88.0
3,104,19.0,B,NaN
4,105,42.0,A,91.0
5,106,35.0,B,85.0
6,107,NaN,NaN,79.0


A right join will contain all the IDs that are in scores, and it will report the groups and age that they are in if those ID's are in the participants table and n/a if not.

In [15]:
right= participants.merge(
    scores,
    on= "id",
    how = "right"
)

In [16]:
right

,id,age,group,score
0,101,24.0,A,81
1,102,31.0,B,94
2,103,27.0,A,88
3,105,42.0,A,91
4,106,35.0,B,85
5,107,NaN,NaN,79


In [17]:
diagnostic = participants.merge(
    scores,
    on="id",
    how="outer",
    indicator=True
)

In [18]:
diagnostic

,id,age,group,score,_merge
0,101,24.0,A,81.0,both
1,102,31.0,B,94.0,both
2,103,27.0,A,88.0,both
3,104,19.0,B,NaN,left_only
4,105,42.0,A,91.0,both
5,106,35.0,B,85.0,both
6,107,NaN,NaN,79.0,right_only


In [19]:
diagnostic["_merge"].value_counts()

_merge
both          5
left_only     1
right_only    1
Name: count, dtype: int64

1. 5 Rows matched
2. 1 occured only in particpants
3. 1 occured only in scores

COME BACK TO 

In [20]:
participants["id"].is_unique
scores["id"].is_unique

True

In [21]:
participants["id"].duplicated().sum()
scores["id"].duplicated().sum()

np.int64(0)

Before we do a one-to-one join we want to make sure that each element in each dataframe is unique so that we do not run into any errors trying to merge an ID in one dataframe to two in another. 

In [22]:
validated = participants.merge(
    scores,
    on="id",
    how="left",
    validate="one_to_one"
)

In [23]:
duplicate_scores = pd.DataFrame({
    "id": [101, 101, 102],
    "score": [81, 83, 94]
})

In [24]:
participants.merge(
    duplicate_scores,
    on="id",
    validate="one_to_one"
)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  id
101 ...

This is an important error as it tells us that the dataframes we are trying to merge are not one-to-one.

In [25]:
visits = pd.DataFrame({
    "id": [1, 1, 2],
    "visit": ["pre", "post", "pre"]
})

treatments = pd.DataFrame({
    "id": [1, 1, 2],
    "treatment": ["A", "B", "A"]
})

In [26]:
visits.merge(
    treatments,
    on="id"
)

,id,visit,treatment
0,1,pre,A
1,1,pre,B
2,1,post,A
3,1,post,B
4,2,pre,A


1. There are 4 rows because ID 1 had 4 different entries.
2. There is no duplicated information in the merged table. For all the instances of ID 1, there are different treatment values.
3. If we were tracking visits to a medical office, and wanted to see which treatments a person received on their pre and post visits, this would be correct. 

In [27]:
fall = pd.DataFrame({
    "id": [1, 2, 3],
    "semester": ["Fall"] * 3,
    "score": [80, 85, 90]
})

spring = pd.DataFrame({
    "id": [4, 5, 6],
    "semester": ["Spring"] * 3,
    "score": [78, 92, 88]
})

In [28]:
all_scores = pd.concat(
    [fall, spring],
    axis=0,
    ignore_index=True
)

In [31]:
fall.merge(
    spring,
    on = "id",
    how = "outer"

)

,id,semester_x,score_x,semester_y,score_y
0,1,Fall,80.0,NaN,NaN
1,2,Fall,85.0,NaN,NaN
2,3,Fall,90.0,NaN,NaN
3,4,NaN,NaN,Spring,78.0
4,5,NaN,NaN,Spring,92.0
5,6,NaN,NaN,Spring,88.0


In [29]:
all_scores

,id,semester,score
0,1,Fall,80
1,2,Fall,85
2,3,Fall,90
3,4,Spring,78
4,5,Spring,92
5,6,Spring,88


In each dataframe for spring and fall, they have all unique values for ID. So if we tried to merge them together we could only produce a table with all info with an outer join. 

In [32]:
fall = pd.DataFrame({
    "id": [1, 2],
    "score": [80, 85]
})

spring = pd.DataFrame({
    "id": [3, 4],
    "score": [90, 92],
    "campus": ["Flagstaff", "Flagstaff"]
})

In [33]:
pd.concat(
    [fall, spring],
    ignore_index=True
)

,id,score,campus
0,1,80,NaN
1,2,85,NaN
2,3,90,Flagstaff
3,4,92,Flagstaff


Because campus is not included in the Fall dataframe, it is recorded as NaN in the resulting table. 

In [34]:
wide = pd.DataFrame({
    "id": [101, 102, 103, 104],
    "group": ["A", "B", "A", "B"],
    "score_pre": [72, 85, 79, 88],
    "score_post": [81, 91, 84, 90]
})

In [35]:
wide

,id,group,score_pre,score_post
0,101,A,72,81
1,102,B,85,91
2,103,A,79,84
3,104,B,88,90


One row represents one participant, and there are no repeated measurments.

In [36]:
long = wide.melt(
    id_vars=["id", "group"],
    value_vars=["score_pre", "score_post"],
    var_name="time",
    value_name="score"
)

In [37]:
long

,id,group,time,score
0,101,A,score_pre,72
1,102,B,score_pre,85
2,103,A,score_pre,79
3,104,B,score_pre,88
4,101,A,score_post,81
5,102,B,score_post,91
6,103,A,score_post,84
7,104,B,score_post,90


1. There are now two rows for each participants.
2. There is now a new column of time, which indicates whether it was a score_pre or score_post.
3. id_vars indicates which rows to keep the same value and repeat down the rows. 
4. value_vars does a similar thing. It keeps score_pre and score_post repeating down the rows.

In [38]:
long["time"] = (
    long["time"]
    .str.replace("score_", "", regex=False)
)

In [39]:
long

,id,group,time,score
0,101,A,pre,72
1,102,B,pre,85
2,103,A,pre,79
3,104,B,pre,88
4,101,A,post,81
5,102,B,post,91
6,103,A,post,84
7,104,B,post,90


In [43]:
long["time"] = (
    long["time"]
    .str.replace("pre","baseline",regex=False)
    .str.replace("post","followup",regex=False)
)
long

,id,group,time,score
0,101,A,baseline,72
1,102,B,baseline,85
2,103,A,baseline,79
3,104,B,baseline,88
4,101,A,followup,81
5,102,B,followup,91
6,103,A,followup,84
7,104,B,followup,90


In [44]:
wide_again = long.pivot(
    index=["id", "group"],
    columns="time",
    values="score"
)

In [45]:
wide_again

,time,baseline,followup
id,group,,
101,A,72,81
102,B,85,91
103,A,79,84
104,B,88,90


In [47]:
wide_again = wide_again.reset_index()
wide_again

time,index,id,group,baseline,followup
0,0,101,A,72,81
1,1,102,B,85,91
2,2,103,A,79,84
3,3,104,B,88,90


Yes it does recover the same information as the OG wide dataset. 

In [50]:
repeated = pd.DataFrame({
    "id": [1, 1, 1, 2, 2],
    "time": ["pre", "pre", "post", "pre", "post"],
    "score": [70, 74, 82, 80, 88]
})
repeated

,id,time,score
0,1,pre,70
1,1,pre,74
2,1,post,82
3,2,pre,80
4,2,post,88


In [49]:
repeated.pivot(
    index="id",
    columns="time",
    values="score"
)

ValueError: Index contains duplicate entries, cannot reshape

This fails because the index has duplicate entries. This violates the assumption that each index corresponds to one value. 

In [52]:
summary = repeated.pivot_table(
    index="id",
    columns="time",
    values="score",
    aggfunc="mean"
)
summary

time,post,pre
id,,
1,82.0,72.0
2,88.0,80.0


Depending what statistical tests we are running, there might be a better option than mean to relay the data. 

In [54]:
clinical = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "age": [24, 31, np.nan, 45, 28, 39],
    "group": ["A", "B", "A", "B", None, "A"],
    "score": [81, np.nan, 77, 92, 88, np.nan]
})
clinical

,id,age,group,score
0,1,24.0,A,81.0
1,2,31.0,B,NaN
2,3,NaN,A,77.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0
5,6,39.0,A,NaN


In [55]:
clinical.isna()

,id,age,group,score
0,False,False,False,False
1,False,False,False,True
2,False,True,False,False
3,False,False,False,False
4,False,False,True,False
5,False,False,False,True


In [56]:
clinical.isna().sum()

id       0
age      1
group    1
score    2
dtype: int64

In [57]:
clinical.isna().mean() * 100

id        0.000000
age      16.666667
group    16.666667
score    33.333333
dtype: float64

In [58]:
clinical.loc[
    clinical["score"].isna()
]

,id,age,group,score
1,2,31.0,B,NaN
5,6,39.0,A,NaN


In [59]:
clinical.loc[
    clinical["score"].notna()
]

,id,age,group,score
0,1,24.0,A,81.0
2,3,NaN,A,77.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0


In [60]:
clinical.loc[
    clinical[["age", "score"]]
    .isna()
    .any(axis=1)
]

,id,age,group,score
1,2,31.0,B,NaN
2,3,NaN,A,77.0
5,6,39.0,A,NaN


In [ ]:
clinical.loc[
    clinical[["age","score"]].notna().all(axis=1)
]


,id,age,group,score
0,1,24.0,A,81.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0


In [66]:
clinical

,id,age,group,score
0,1,24.0,A,81.0
1,2,31.0,B,NaN
2,3,NaN,A,77.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0
5,6,39.0,A,NaN


In [67]:
complete_score = clinical.dropna(
    subset=["score"]
)

In [71]:
len(clinical)

6

In [70]:
len(complete_score)

4

In [78]:
complete_cases = clinical.dropna(
    subset=["age", "score"]
)
complete_cases


,id,age,group,score
0,1,24.0,A,81.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0
